# Chapter 7 — Report and resolve exact requests

Source candidate · CONVERGING · checkpoint-bound evidence

> **Source candidate / CONVERGING.** These cells describe the Chapter’s
> model and operations; their presence is not execution or scientific
> evidence. The website runs no kernels or solvers, and the generated
> Notebook remains zero-output. See the earlier diagram lessons for
> circuit review.

One parameterized resonator Plan can answer more than one question. This
Chapter keeps the physical declaration separate from the Direct and root
requests it collects into a report, then shows how the exact root
request is resolved after a fresh-kernel restart.

## Lesson 7.1 — Build one parameterized model

### A. Declare the complete model

This first route is complete from a fresh kernel: it repeats the
physical declaration before constructing any numerical request.

The sequence remains root/child ownership, public parameters, native
parts, local parallel structure, one exposed terminal, then root wiring.

In [ ]:
from scnsim import CircuitPlan, ParameterDefinitions, ParameterSpec, components, units as u

inputs = ParameterDefinitions(id="readout_design")
plan = CircuitPlan(id="primitive_resonator")
resonator = plan.subsystem(id="resonator")

Independent refs are bound by the following child physical fields on the
real capacitor and inductor.

In [ ]:
capacitance = inputs.parameter(
    id="capacitance",
    baseline=110.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)
inductance = inputs.parameter(
    id="inductance",
    baseline=5.8 * u.nH,
    spec=ParameterSpec(unit=u.nH),
)

These two ordinary ElementUses are ready for the local parallel
relation.

In [ ]:
capacitor = resonator.add(
    components.capacitor(id="capacitor", capacitance=capacitance)
)
inductor = resonator.add(
    components.inductor(id="inductor", inductance=inductance)
)

The local branches share their terminal bus and the child’s ground.

In [ ]:
resonator_bus = resonator.bus(id="terminal")
parallel_lc = resonator.parallel(
    id="parallel_lc",
    start=resonator_bus,
    branches=((capacitor,), (inductor,)),
    end=resonator.ground,
)

In [ ]:
terminal = resonator.expose_pin(id="terminal", at=resonator_bus)

The root uses this published terminal rather than either private leaf.

In [ ]:
signal_bus = plan.bus(id="signal_boundary")
resonator_root_bus = plan.bus(id="resonator_node")
coupling_cap = plan.add(
    components.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
coupling = plan.series(
    id="coupling",
    start=signal_bus,
    elements=(coupling_cap,),
    end=resonator_root_bus,
)
plan.link(
    id="resonator_terminal",
    endpoints=(resonator_root_bus, terminal),
)
signal_port = plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
resonator_node = resonator_root_bus.node

`signal_port` and `resonator_node` are the root-facing handles used by
the request-only cell.

## Lesson 7.2 — Report and resolve the named questions

### B. Construct the View, specifications, and request

This cell deliberately performs no numerical execution. It is safe to
rerun after rebuilding the declaration while the execution cells remain
skipped.

In [ ]:
from scnsim import (
    CircuitRun,
    DiagonalRootSpec,
    DirectSolveSpec,
    ParameterSet,
    ReductionPipeline,
)

workspace = "workspaces/primitive-course"
run = CircuitRun(plan=plan, workspace=workspace)
direct_spec = DirectSolveSpec(
    frequencies=[5.5, 6.0, 6.5, 7.0] * u.GHz
)
quantity_view = run.original.reduce(
    ReductionPipeline().retain(resonator_node)
)
root_spec = DiagonalRootSpec(
    coordinate=resonator_node,
    root_hint=6.0 * u.GHz,
)
baseline_parameters = ParameterSet()

`direct_spec`, `quantity_view`, and `root_spec` can be rebuilt without
running any numerical operation.

### C. Execute and inspect the ordinary route

The ordinary route explicitly supplies its two Results to the report.

In [ ]:
from scnsim import ReportSpec, Theme

direct = run.solve(run.original, direct_spec, parameters=baseline_parameters)
root = run.evaluate(quantity_view, root_spec, parameters=baseline_parameters)
report = run.build_report(
    ReportSpec(inputs=(direct, root), theme=Theme.AUTO)
)

The next cell presents the two explicit Results and their derived
report.

In [ ]:
direct.s.show(magnitude="db")
root.show()
report.show()

### D. Restart and resolve the exact request

For the ordinary route, run sections A, B, and C. For an exact-request
restart, restart the kernel; rerun every section-A declaration cell and
the section-B request-only cell; skip both section-C
execution/inspection cells; then run the resolve-only cell below. Those
same rebuilt cells provide the fresh `run`, `quantity_view`, and
`root_spec`.

### E. Resolve and inspect the rebuilt exact request

In [ ]:
resolved = run.resolve(
    quantity_view,
    root_spec,
    parameters=baseline_parameters,
)
resolved.show()

Neither operation performs a new execution: a report derives
presentation from explicit Results, and `resolve()` reads the exact
verified stored request evidence.

[Previous](06_optimize_primitive.qmd) · [Next](08_create_library.qmd)